# THINGS Odd-One-Out: Human–Model Alignment — Colab Runner

Runs the **full pipeline** on a Colab GPU — everything is scripted, including the
CC0 images (no login or agreement). The smoke-test section validates Phases 3–5
**without images** using synthetic features; section 4 is the real run.

**Tip:** Runtime → Change runtime type → GPU (T4).

## 1. Clone + install

In [ ]:
import os
os.chdir('/content')   # always start from /content so re-running never nests clones
if not os.path.isdir('VisionModal-HumalObjectSimilarity/.git'):
    !git clone https://github.com/Sudarssan-N/VisionModal-HumalObjectSimilarity.git
%cd /content/VisionModal-HumalObjectSimilarity
!git pull --ff-only    # always run the latest pushed code
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Download + prepare behavioral data (scripted, ~412 MB)

In [ ]:
!python src/download_data.py
!cd data/raw && unzip -q -o osfstorage-archive.zip -d extracted
!cd data/raw/extracted && unzip -q -o full_triplet_dataset.zip -d triplets
!python src/prepare_data.py

## 3. Smoke test — validate Phases 3–5 WITHOUT images

Fabricates features that are a noisy linear function of the real human
embedding, then runs the real training/transfer/analysis scripts. Confirms the
pipeline works on this machine before investing in feature extraction.

In [ ]:
!python src/smoke_test.py --force

## 4. Real run: images → features → results

The CC0 reference images (one per concept, 1.1 GB) download automatically — no
login or agreement needed.

In [ ]:
# Download + organize the 1,854 CC0 images (1.1 GB)
!python src/download_data.py --images
!python src/organize_images.py data/raw/images_cc0
!python src/data.py

In [ ]:
# Phase 1 + 2: extract embeddings (frozen, GPU), zero-shot baseline
!python src/extract_features.py
!python src/zeroshot_eval.py

In [ ]:
# Phase 3: alignment — multi-seed on FULL data so Table 1 reports mean±std (CIs for Q2)
!SEEDS=0,1,2 python src/train_transform.py
# Phase 4-5: cross-model transfer (+ Procrustes basis-alignment control) and analysis
!python src/transfer.py
!python src/analysis.py

## 4b. Transfer confound check — PCA-basis alignment (Procrustes control)

The cross-model transfer in §4 compares transforms in a shared PCA space, but each
model is PCA-reduced **independently**, so two models' shared-d axes are not in a
common coordinate frame. Applying `W_src` to a target then pays a penalty for
*basis* mismatch on top of any genuine model-specific structure — which can make
shared structure look model-specific.

`src/transfer.py` now also computes a **basis-aligned** transfer matrix: for each
(source, target) pair it rotates the target's PCA basis into the source's frame via
**orthogonal Procrustes** over the 1,854 shared concepts (no human labels), then
applies `W_src`. Orthogonal alignment only rotates/reflects — it cannot rescale or
distort the geometry, so it removes the basis confound without doing any of the
alignment work. The cell below compares both matrices and their recovery fractions.

In [ ]:
# === Transfer confound check: independent-PCA (raw) vs Procrustes basis-aligned ===
# src/transfer.py now emits BOTH transfer matrices + recovery fractions. The
# basis-aligned recovery is the load-bearing number for "transforms are
# model-specific": independently-fit PCA bases can make genuinely shared structure
# LOOK model-specific, so we rotate each target into the source's PCA frame
# (orthogonal Procrustes over the shared concepts, no human labels) before transfer.

# Tip: re-run transfer alone to iterate on dimensionality, e.g.:
# !SHARED_DIM=256 python src/transfer.py

import json
from pathlib import Path
from IPython.display import Image, display

tr = json.loads(Path('results/transfer.json').read_text())
names = tr['names']

def show(M, title):
    print(title)
    print('              ' + ''.join(f'{n[:10]:>11}' for n in names))
    for i, s in enumerate(names):
        print(f'{s[:12]:12}  ' + ''.join(f'{M[i][j]:11.3f}' for j in range(len(names))))

show(tr['transfer_matrix'], 'RAW transfer (independent PCA)')
print()
show(tr.get('transfer_matrix_basis_aligned', tr['transfer_matrix']),
     'BASIS-ALIGNED transfer (Procrustes control)')

print('\nRecovery fraction rho_t  (raw  ->  basis-aligned):')
rho, rho_ba = tr['recovery_fraction'], tr['recovery_fraction_basis_aligned']
for n in names:
    print(f'  {n:16} {rho[n]:+.3f}  ->  {rho_ba[n]:+.3f}')
print(f'  {"MEAN":16} {tr["mean_recovery"]:+.3f}  ->  {tr["mean_recovery_basis_aligned"]:+.3f}')

print('\nRead-out:')
print('  basis-aligned recovery stays LOW  -> weak transfer is genuine model-specificity')
print('  basis-aligned recovery jumps HIGH -> raw weak transfer was a PCA-basis artifact')
display(Image('results/transfer_heatmap.png'))

In [ ]:
# Assemble all results into a single markdown report
!python src/make_report.py

## 5. Robustness (seeds / λ sweep / image-disjoint split)

Confirms the aligned numbers are stable and not a leakage artifact. Adds a
robustness section to `results/REPORT.md`.

In [ ]:
!python src/run_robustness.py
!python src/make_report.py

## 6. Compile the paper (PDF)

Builds the NeurIPS-format preprint with the real figures. Download `neurips_2024.sty`
into `paper/` for the official format (optional — falls back to a clean article style).

In [ ]:
!apt-get -qq install texlive-latex-extra texlive-bibtex-extra texlive-fonts-recommended >/dev/null
import os; os.makedirs('paper/figures', exist_ok=True)
!cp results/transfer_heatmap.png results/rsa.png paper/figures/ 2>/dev/null || true
%cd paper
!pdflatex -interaction=nonstopmode main >/dev/null && bibtex main >/dev/null && pdflatex -interaction=nonstopmode main >/dev/null && pdflatex -interaction=nonstopmode main >/dev/null
%cd ..
from google.colab import files
files.download('paper/main.pdf')